# Week 82, Day 3: Reliability

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w082d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
minutes_per_year = 365 * 24 * 60
for nines in (2, 3, 4, 5):
    availability = 1 - 10 ** -nines
    print(f"{availability:.5%}: {minutes_per_year * (1 - availability):.0f} minutes down a year")

In [ ]:
slo = 0.999
for errors, total, window in [(30, 10_000, "5 min"), (1_200, 100_000, "1 hour"), (90, 100_000, "1 hour")]:
    burn = (errors / total) / (1 - slo)
    print(f"{window}: error rate {errors / total:.2%}, burn rate {burn:.1f}")

> **Key idea.** Availability multiplies in series and combines as 1 - Π(1 - a) in parallel (if failures are independent); each nine is ten times less downtime. SLIs measure, SLOs set targets over a window, SLAs are contracts; the error budget (1 - SLO) is spent on risk until it runs out. Page on burn rate (error rate / (1 - SLO)), high in both a long and a short window (14.4 for an hour burns 2% of a 30-day budget). Prevent cascades: timeouts on every call and deadlines that shrink as a request proceeds; retries only for retryable errors on idempotent operations, with exponential backoff, full jitter and a retry budget; circuit breakers (closed, open after repeated failures, half-open trial); bulkheads per dependency; fallbacks to cached or default answers; load shedding. Test failures on purpose, roll out gradually, restore first, and write blameless postmortems.

### Exercise: Nines

Write `series(*availabilities)` (all must work: the product) and `parallel(*availabilities)` (any one is enough: `1 - product of (1 - a)`), each rounded to 6 places. Then `downtime_minutes(availability, days=365)`: the allowed downtime in minutes over that many days, rounded to 1 place. And `nines(availability)`: the number of leading nines in its percentage form, so `0.999` is 3, `0.9995` is 3 and `0.99` is 2 (use `-log10(1 - a)` rounded down, after a tiny nudge such as `+ 1e-9` for floating-point error; a perfect `1.0` raises `ValueError("nothing is 100% available")`).

In [ ]:
import math

def series(*availabilities):
    ___

def parallel(*availabilities):
    ___

def downtime_minutes(availability, days=365):
    ___

def nines(availability):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_nines():\n    \"series lowers, parallel raises\"\n    assert series(0.999, 0.999, 0.999) == 0.997003 and series(0.9999, 0.99) == 0.989901\n    assert parallel(0.99, 0.99) == 0.9999 and parallel(0.9, 0.9, 0.9) == 0.999\n    assert series(parallel(0.99, 0.99), 0.999) == 0.9989\n    assert downtime_minutes(0.999) == 525.6 and downtime_minutes(0.9999, 30) == 4.3\n    assert nines(0.999) == 3 and nines(0.9995) == 3 and nines(0.99) == 2 and nines(0.99999) == 5 and nines(0.5) == 0\n    try:\n        nines(1.0)\n    except ValueError as e:\n        assert str(e) == \"nothing is 100% available\"")

### Exercise: Error budgets

Write `budget(slo, total_requests)`: how many requests may fail in the window (rounded down). `remaining(slo, good, total)`: the fraction of the error budget left, as `1 - failures / allowed_failures` rounded to 3 places (it can go negative when the budget is overspent; with `total` 0, return `1.0`). And `policy(slo, good, total)`: `"ship"` if more than 25% of the budget remains, `"careful"` if any remains, else `"freeze"` (no risky changes until it recovers).

In [ ]:
import math

def budget(slo, total_requests):
    ___

def remaining(slo, good, total):
    ___

def policy(slo, good, total):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_budget():\n    \"spend it wisely\"\n    assert budget(0.999, 10_000_000) == 10_000 and budget(0.99, 150) == 1\n    assert remaining(0.999, 9_996_000, 10_000_000) == 0.6 and remaining(0.999, 9_985_000, 10_000_000) == -0.5\n    assert remaining(0.999, 0, 0) == 1.0\n    assert policy(0.999, 9_996_000, 10_000_000) == \"ship\" and policy(0.999, 9_991_000, 10_000_000) == \"careful\"\n    assert policy(0.999, 9_990_000, 10_000_000) == \"freeze\"")

### Exercise: Burn-rate alerts

Write `burn_rate(errors, total, slo)`: the error rate divided by `1 - slo`, rounded to 2 places (`0.0` with no traffic). Then `check_alerts(windows, slo)`: `windows` maps window names (`"5m"`, `"30m"`, `"1h"`, `"6h"`) to `(errors, total)`. Return a list of alerts using the standard multi-window rules: `"page"` if the burn rate is at least **14.4** in both `"1h"` and `"5m"`; `"page"` if at least **6** in both `"6h"` and `"30m"` (once: don't add `"page"` twice); `"ticket"` if at least **1** in `"6h"` but no page was raised. Return `[]` if all is well.

In [ ]:
def burn_rate(errors, total, slo):
    ___

def check_alerts(windows, slo):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_burn():\n    \"fast burns page, slow burns ticket\"\n    assert burn_rate(30, 10_000, 0.999) == 3.0 and burn_rate(0, 0, 0.999) == 0.0\n    fast = {\"5m\": (200, 10_000), \"30m\": (900, 60_000), \"1h\": (1_800, 120_000), \"6h\": (2_000, 720_000)}\n    assert check_alerts(fast, 0.999) == [\"page\"]\n    over = {\"5m\": (5, 10_000), \"30m\": (900, 60_000), \"1h\": (1_800, 120_000), \"6h\": (2_000, 720_000)}\n    assert check_alerts(over, 0.999) == [\"ticket\"], \"a burst that has already stopped shouldn't page\"\n    slow = {\"5m\": (70, 10_000), \"30m\": (420, 60_000), \"1h\": (800, 120_000), \"6h\": (4_600, 720_000)}\n    assert check_alerts(slow, 0.999) == [\"page\"]\n    leak = {\"5m\": (15, 10_000), \"30m\": (90, 60_000), \"1h\": (180, 120_000), \"6h\": (1_000, 720_000)}\n    assert check_alerts(leak, 0.999) == [\"ticket\"]\n    calm = {\"5m\": (1, 10_000), \"30m\": (5, 60_000), \"1h\": (10, 120_000), \"6h\": (60, 720_000)}\n    assert check_alerts(calm, 0.999) == []")

### Exercise: A circuit breaker

Write a class `CircuitBreaker(clock, failure_threshold, reset_timeout)` with `state` (`"closed"`, `"open"` or `"half-open"`). `call(func)`: when **open**, if `reset_timeout` seconds have passed since it opened, move to **half-open** and let this call through as a trial; otherwise raise `CircuitOpenError("circuit open")` without calling. When the call raises, count a consecutive failure; in half-open, or on reaching `failure_threshold` failures, **open** (recording the time) and re-raise. When it succeeds, reset the failure count, **close** the circuit, and return the result. Define `CircuitOpenError` as a subclass of `Exception`.

In [ ]:
class CircuitOpenError(Exception):
    pass

class CircuitBreaker:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_breaker():\n    \"closed, open, half-open\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    b = CircuitBreaker(clock, failure_threshold=3, reset_timeout=30)\n    calls = []\n    def down():\n        calls.append(1)\n        raise ConnectionError(\"down\")\n    for _ in range(3):\n        try:\n            b.call(down)\n        except ConnectionError:\n            pass\n    assert b.state == \"open\" and len(calls) == 3\n    try:\n        b.call(down)\n    except CircuitOpenError as e:\n        assert str(e) == \"circuit open\"\n    assert len(calls) == 3, \"an open circuit doesn't call\"\n    clock.t = 30\n    try:\n        b.call(down)\n    except ConnectionError:\n        pass\n    assert b.state == \"open\" and len(calls) == 4\n    clock.t = 59\n    try:\n        b.call(lambda: \"ok\")\n    except CircuitOpenError:\n        pass\n    else:\n        raise AssertionError(\"still cooling down\")\n    clock.t = 60\n    assert b.call(lambda: \"ok\") == \"ok\" and b.state == \"closed\"\n    try:\n        b.call(down)\n    except ConnectionError:\n        pass\n    assert b.state == \"closed\", \"one failure isn't enough once closed again\"")

### Exercise: Backoff with jitter, on a budget

Write `full_jitter(attempt, base, cap, rng)`: the wait before retry number `attempt` (from 0): a random number between 0 and `min(cap, base * 2 ** attempt)`, using `rng.uniform(0, ceiling)`. Then a class `RetryBudget(ratio, minimum=10)`: `request()` counts a request; `can_retry()` returns `True` (and counts a retry) only if retries so far would stay at most `ratio` of requests, or fewer than `minimum` retries have been made in total; otherwise `False`.

In [ ]:
def full_jitter(attempt, base, cap, rng):
    ___

class RetryBudget:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_jitter():\n    \"spread out, capped\"\n    import random\n    rng = random.Random(42)\n    waits = [full_jitter(a, 1.0, 30.0, rng) for a in range(8)]\n    assert all(0 <= w <= min(30, 2 ** a) for a, w in enumerate(waits))\n    assert len({round(w, 6) for w in waits}) == 8\n    class Fixed:\n        def uniform(self, lo, hi):\n            return hi\n    assert [full_jitter(a, 0.5, 4.0, Fixed()) for a in range(5)] == [0.5, 1.0, 2.0, 4.0, 4.0]\n\ndef test_budget():\n    \"retries can't multiply load\"\n    b = RetryBudget(0.1, minimum=2)\n    assert b.can_retry() and b.can_retry() and not b.can_retry()\n    for _ in range(30):\n        b.request()\n    assert b.can_retry() is True and b.can_retry() is False")

### Exercise: Deadlines that shrink

Write a class `Deadline(clock, seconds)`: `remaining()` is the time left (never below 0), and `expired()` whether none is left. `timeout_for(call_timeout)` is the smaller of the call's own timeout and the time remaining. Then `run_steps(deadline, steps, minimum=0.05)`: `steps` is a list of `(name, timeout, duration, clock_advance)` where `clock_advance` is a function taking the seconds to advance the test clock. For each step, if less than `minimum` remains, add `"NAME skipped"` and stop (the caller has given up); otherwise use `t = deadline.timeout_for(timeout)`: if the step's `duration` is more than `t`, advance the clock by `t` and add `"NAME timed out"` and stop; else advance by `duration` and add `"NAME ok"`. Return the list.

In [ ]:
class Deadline:
    ___

def run_steps(deadline, steps, minimum=0.05):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_deadline():\n    \"no call outlives the request\"\n    class Clock:\n        t = 0.0\n        def __call__(self):\n            return self.t\n        def advance(self, s):\n            self.t += s\n    clock = Clock()\n    d = Deadline(clock, 2.0)\n    assert d.remaining() == 2.0 and d.timeout_for(5.0) == 2.0 and d.timeout_for(0.5) == 0.5\n    steps = [(\"auth\", 1.0, 0.1, clock.advance), (\"cart\", 1.0, 0.4, clock.advance), (\"payments\", 3.0, 1.9, clock.advance), (\"email\", 1.0, 0.1, clock.advance)]\n    assert run_steps(d, steps) == [\"auth ok\", \"cart ok\", \"payments timed out\"]\n    assert d.expired() and d.remaining() == 0\n    clock.t = 0.0\n    d2 = Deadline(clock, 1.0)\n    clock.t = 0.97\n    assert run_steps(d2, [(\"late\", 1.0, 0.01, clock.advance)]) == [\"late skipped\"]")

### Bug hunt: Redundancy that made things worse

Two replicas, either of which can serve requests, are combined as if both were needed, so the estimate claims the redundant pair is **less** available than one replica: `pair: 98.01%`. Combine them in parallel, so it prints `pair: 99.99%`.

In [ ]:
replica = 0.99
pair = replica * replica
print(f"pair: {pair:.2%}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"parallel\"\n    expect_output(\"pair: 99.99%\")")

### Bug hunt: A thundering retry

After an outage, 1,000 clients all retry after exactly the backoff time, so they hit the recovering service in the same instant: `busiest second: 1000 retries`. Use full jitter (a random wait between 0 and the backoff, from the seeded `rng`), so the retries spread out and the busiest second has far fewer: `busiest second: under 200 retries`.

In [ ]:
import random
from collections import Counter

rng = random.Random(7)
backoff = 8.0
retry_times = [backoff for _ in range(1000)]
busiest = max(Counter(int(t) for t in retry_times).values())
print(f"busiest second: {busiest} retries" if busiest >= 200 else "busiest second: under 200 retries")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"jitter\"\n    expect_output(\"busiest second: under 200 retries\")")

### Bug hunt: A circuit that never closed

This breaker opens after three failures but never checks whether its cool-down has passed, so even after the dependency recovers every call is refused: `at 100s: refused`. Let one trial call through once 30 seconds have passed since it opened, closing the circuit if it works, so it prints `at 100s: ok`.

In [ ]:
class Breaker:
    def __init__(self):
        self.open, self.failures, self.opened_at = False, 0, None

    def call(self, func, now):
        if self.open:
            return "refused"
        try:
            func()
        except Exception:
            self.failures += 1
            if self.failures >= 3:
                self.open, self.opened_at = True, now
            return "failed"
        self.failures = 0
        return "ok"

b = Breaker()
def broken():
    raise ConnectionError
for t in range(3):
    b.call(broken, t)
print(f"at 100s: {b.call(lambda: None, 100)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"half-open after the cool-down\"\n    expect_output(\"at 100s: ok\")")

### Puzzle: A careful call to a dependency

Arrange a call that respects the request's deadline, goes through a circuit breaker, and falls back to a cached value. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # minutes of downtime a 99.9% SLO allows over 30 days (rounded to a whole number)
q = ___   # the burn rate when the error rate is 1% and the SLO is 99.9%
r = ___   # the breaker state that lets one trial call through
s = ___   # True or False: retry a POST that returned 400
t = ___   # the availability of two independent 90% replicas in parallel

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 43 and abs(q - 10) < 1e-9 and r == \"half-open\", \"p, q and r\"\n    assert s is False and abs(t - 0.99) < 1e-9, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a resilient client

Write a class `ResilientClient(clock, call, breaker, cache, rng, max_attempts=3, base=0.1, cap=1.0)`: `call(key, timeout)` fetches a value from a dependency (it returns the value, or raises `TimeoutError` or `ConnectionError`, which are retryable, or `ValueError`, which isn't); `breaker` is a `CircuitBreaker` (given); `cache` is a dict used as a fallback. `get(key, deadline)` (a `Deadline`, given) returns `(value, source)`: try up to `max_attempts` times through the breaker with `timeout = deadline.timeout_for(1.0)`; on success store the value in the cache and return `(value, "live")`. On a retryable error, if attempts remain, wait `full_jitter(attempt, base, cap, rng)` (the given function) by appending the wait to `self.waits` and advancing the clock with `clock.advance(wait)`, but give up early if the deadline has less than 0.05 seconds left after waiting. A `CircuitOpenError` means stop trying at once. A `ValueError` is re-raised immediately (no retries, no fallback: a bad request is the caller's problem). When giving up, return `(cache[key], "stale")` if cached, else raise `LookupError(f"no value for {key}")`. Record each outcome in `self.log` as `"live"`, `"stale"`, `"error"`.

In [ ]:
class CircuitOpenError(Exception):
    pass

class CircuitBreaker:
    def __init__(self, clock, failure_threshold, reset_timeout):
        self.clock, self.threshold, self.reset_timeout = clock, failure_threshold, reset_timeout
        self.state, self.failures, self.opened_at = "closed", 0, None
    def call(self, func):
        if self.state == "open":
            if self.clock() - self.opened_at >= self.reset_timeout:
                self.state = "half-open"
            else:
                raise CircuitOpenError("circuit open")
        try:
            result = func()
        except Exception:
            self.failures += 1
            if self.state == "half-open" or self.failures >= self.threshold:
                self.state, self.opened_at = "open", self.clock()
            raise
        self.failures, self.state = 0, "closed"
        return result

class Deadline:
    def __init__(self, clock, seconds):
        self.clock, self.ends = clock, clock() + seconds
    def remaining(self):
        return max(0.0, self.ends - self.clock())
    def timeout_for(self, call_timeout):
        return min(call_timeout, self.remaining())

def full_jitter(attempt, base, cap, rng):
    return rng.uniform(0, min(cap, base * 2 ** attempt))

class ResilientClient:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Clock:\n    def __init__(self):\n        self.t = 0.0\n    def __call__(self):\n        return self.t\n    def advance(self, s):\n        self.t += s\n\nclass Half:\n    def uniform(self, lo, hi):\n        return hi / 2\n\ndef _client(behaviour, threshold=5):\n    clock = Clock()\n    calls = []\n    def call(key, timeout):\n        calls.append((key, round(timeout, 3)))\n        outcome = behaviour(len(calls))\n        if isinstance(outcome, Exception):\n            raise outcome\n        return outcome\n    breaker = CircuitBreaker(clock, threshold, 30)\n    return ResilientClient(clock, call, breaker, {}, Half()), clock, calls\n\ndef test_live_and_retries():\n    \"retries with jitter, then success\"\n    client, clock, calls = _client(lambda n: TimeoutError() if n < 3 else {\"GBP\": 1})\n    assert client.get(\"rates\", Deadline(clock, 2.0)) == ({\"GBP\": 1}, \"live\")\n    assert client.waits == [0.05, 0.1] and len(calls) == 3 and client.cache[\"rates\"] == {\"GBP\": 1} and client.log == [\"live\"]\n\ndef test_stale_and_errors():\n    \"fallbacks, bad requests and empty caches\"\n    client, clock, calls = _client(lambda n: {\"v\": 1} if n == 1 else ConnectionError())\n    client.get(\"rates\", Deadline(clock, 2.0))\n    assert client.get(\"rates\", Deadline(clock, 2.0)) == ({\"v\": 1}, \"stale\") and len(calls) == 4\n    try:\n        client.get(\"other\", Deadline(clock, 2.0))\n    except LookupError as e:\n        assert str(e) == \"no value for other\"\n    assert client.log == [\"live\", \"stale\", \"error\"]\n    bad, clock2, calls2 = _client(lambda n: ValueError(\"bad currency\"))\n    try:\n        bad.get(\"rates\", Deadline(clock2, 2.0))\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"ValueError isn't retried or hidden\")\n    assert len(calls2) == 1\n\ndef test_breaker_and_deadline():\n    \"an open circuit and a short deadline stop early\"\n    client, clock, calls = _client(lambda n: {\"v\": n} if n == 1 else ConnectionError(), threshold=2)\n    client.get(\"k\", Deadline(clock, 2.0))\n    assert client.get(\"k\", Deadline(clock, 2.0)) == ({\"v\": 1}, \"stale\") and len(calls) == 3\n    assert client.get(\"k\", Deadline(clock, 2.0)) == ({\"v\": 1}, \"stale\") and len(calls) == 3, \"open circuit: no calls\"\n    slow, clock3, calls3 = _client(lambda n: TimeoutError())\n    slow.cache[\"k\"] = \"old\"\n    assert slow.get(\"k\", Deadline(clock3, 0.08)) == (\"old\", \"stale\") and len(calls3) == 1")

## Warm-up

Each extra **nine** of availability cuts the allowed downtime by ten, and costs much more than the one before. 99.9% allows almost nine hours a year; 99.999% allows five minutes, which is less than one bad deploy. Reliability is a choice with a price, so the first job is deciding how much you need.

## Availability arithmetic

**Availability** is the fraction of time (or of requests) a service works. Combining components:

- **In series** (a request needs **all** of them: load balancer, app, database), availabilities **multiply**: three components at 99.9% give `0.999³ = 99.7%`. Every dependency you add lowers the ceiling.
- **In parallel** (any one **will do**: two replicas behind a load balancer), the system fails only if **all** fail: `1 - (1 - a)ⁿ`. Two 99% replicas give 99.99%, if their failures are independent, which is the hard part: replicas in one data centre, on one software version, fail together.

So redundancy is how reliable systems are built from unreliable parts, and every shared dependency (one database, one DNS provider, one region) is a place where redundancy quietly stops.

## SLOs and error budgets

Three terms, used precisely:

- An **SLI** (service level **indicator**) is a measurement: the share of requests that succeed, or that succeed in under 300 ms.
- An **SLO** (objective) is a target for an SLI over a window: "99.9% of requests succeed, over 30 days".
- An **SLA** (agreement) is a contract with consequences (refunds) when it's missed, usually looser than the SLO.

The SLO's complement is the **error budget**: with 99.9% over 30 days, 0.1% of requests (or about 43 minutes of full outage) may fail. The budget turns reliability into a decision rule: while there's budget left, ship features and take risks; when it's spent, slow down and fix reliability. Aiming for 100% is a mistake: it's impossible, and trying makes every change too risky to make.

**Alerting on the budget.** "Error rate above 0.1%" pages someone for every small blip, and misses a slow leak. Better: page on the **burn rate**, how fast the budget is being used: the error rate divided by the allowed error rate (`1 - SLO`). A burn rate of 1 uses exactly the budget over the window; **14.4** would use 2% of a 30-day budget in one hour, which deserves a page now. To avoid paging for a burst that's already over, require the high burn rate over **two windows**: a long one (an hour: is it significant?) and a short one (five minutes: is it still happening?).

## Stopping failures spreading

Most big outages are **cascading** failures: one slow dependency makes callers wait, their threads and connections fill up, they slow down, and their callers do the same, until everything is down. The defences:

- **Timeouts on every call**, and **deadlines** for whole requests: if the user's request must finish in 2 seconds and 1.5 have gone, a downstream call gets at most 0.5, and if less than some minimum remains, don't make the call at all. The deadline travels with the request (gRPC sends it as a header), so work that nobody is waiting for any more is never started.
- **Retries, carefully**: only for errors that might succeed next time (timeouts, `503`, connection resets; never `400`); only for idempotent operations (Week 77); with **exponential backoff** and **jitter**. With "full jitter", each wait is a **random** time between 0 and the backoff ceiling, so a thousand clients that failed together don't all retry at the same instant. And with a **retry budget**: allow retries only up to, say, 10% of requests, so retries can't multiply the load on a service that's already struggling.
- **Circuit breakers**: track a dependency's recent failures; after too many in a row, **open** the circuit and fail calls immediately (no waiting, no load on the struggling service) for a cool-down period; then go **half-open** and let one trial call through: success **closes** the circuit, failure opens it again.
- **Bulkheads**: separate pools of connections or workers per dependency (named after a ship's watertight compartments), so one slow dependency can only exhaust its own pool.
- **Fallbacks and graceful degradation**: when a non-essential dependency fails, serve something useful anyway: a cached value (Week 81's stale-while-revalidate), a default, the page without its recommendations panel.
- **Load shedding** (last week's Forge bonus): refuse excess work quickly rather than serving everyone slowly.

## When it goes wrong anyway

Practices that make failures rarer and shorter:

- **Test failure on purpose**: chaos engineering kills instances and injects latency in controlled experiments, so the fallbacks are known to work before they're needed.
- **Roll out gradually** (Week 77): canaries and staged rollouts limit how many users a bad change reaches, and quick rollback limits for how long.
- **Incident response**: one person coordinates, someone communicates, the priority is restoring service (roll back first, debug later).
- **Blameless postmortems**: afterwards, write down what happened, why the system allowed it, and what will change, without blaming people: people make mistakes, and systems should make them hard to make and cheap to recover from.

### On your own computer

`tenacity` (`pip install tenacity`) adds retries with backoff and jitter to any function; `pybreaker` provides circuit breakers; `httpx` and `requests` take a `timeout` on every call (and have **no** timeout by default, so always pass one). The SRE books from Google (free to read online) are the classic source for SLOs and error budgets.

## Practice

## Quiz

## Daily challenge

## Before you go

Find the status page of a service you depend on (most large services publish one) and look at its incident history: you'll recognise cascading failures, bad deploys and dependency outages from today. Tomorrow: **design interview practice**, putting the last two weeks together under time pressure.